## Validation of Four Potential Models

This notebook explores the validity of the models described in the following notebooks:
- 03_baseline_logistical_regression_model.ipynb
- 04_baseline_randomforest_model.ipynb

### Call Libraries and Modules

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

root = Path().resolve()
while not (root/"src").exists():
    root = root.parent
sys.path.append(str(root))

from src.data_processing import pull_data, create_target, select_features
from src.training_functions import splitting_data, scaling_data
from src.validation_metrics import cross_validation

### Load and Process Data

In [2]:
df = pull_data("data/atlas-higgs-challenge-2014-v2.csv")
df = create_target(df)
x_data, y_data = select_features(df, ["PRI_met", "DER_pt_tot", "DER_mass_jet_jet", "DER_pt_h"], "Target")

### Set up Cross Validation

Due to the dataset only being 818000 records, a stratified K-Fold will be applied. The cross validation function in validation_metrics.py creates 10 K-Folds, reducing the data from each to ~80000 records. This is enough to test the consistency of the model without impacting the training and testing processes.

In [3]:
pipeline = Pipeline([
    ("scalar", StandardScaler()),
    ("classifier", LogisticRegression())
])
lr = cross_validation(pipeline, x_data, y_data, "Logistic Regression")

In [4]:
rfc = cross_validation(RandomForestClassifier(n_estimators=25), x_data, y_data, "Random Forest Classifier", 5)

Due to the processing time, the regession forest only used 5 K-Folds, and reduced the number of trees from 100 to 25.

### Collect Results

In [5]:
model_information = []
model_information.append(lr)
model_information.append(rfc)
df = pd.DataFrame(model_information)
print(df)

                             Average Accuracy  Accuracy STD  Maximum Accuracy  \
0       Logistic Regression          0.681078      0.001023          0.682929   
1  Random Forest Classifier          0.680245      0.000577          0.681127   

   Minimum Accuracy  Average ROC AUC  ROC AUC STD  Maximum ROC AUC  \
0          0.679051         0.678342     0.002043         0.682811   
1          0.679544         0.692452     0.001662         0.694897   

   Minimum ROC AUC  Average Log Loss  Log Loss STD  Maximum Log Loss  \
0         0.675731         -0.603889      0.000745         -0.602287   
1         0.689727         -0.886763      0.007772         -0.877894   

   Minimum Log Loss  
0         -0.604894  
1         -0.900511  


### Results and Conclusions

Both models produces a similary in regards to accuracy. 68% suggest the features had a slight impact on differentiating signals from background, however further exploration of other features and feature engineering will be required. The Random Forest Classifier performed marginally better in regards to ROC AUC while 

In [3]:
table = pd.DataFrame({"Logistic Regression": [f"0.681078 ± 0.001023", f"0.678342 ± 0.002043", f"-0.603889 ± 0.000745"], "Random Forest Classifier": [f"0.680245 ± 0.000577", f"0.692452 ± 0.001662", f"-0.886763 ± 0.007772"]}, index = (["Accuracy", "ROC AUC", "Log Loss"]))
print(table)

           Logistic Regression Random Forest Classifier
Accuracy   0.681078 ± 0.001023      0.680245 ± 0.000577
ROC AUC    0.678342 ± 0.002043      0.692452 ± 0.001662
Log Loss  -0.603889 ± 0.000745     -0.886763 ± 0.007772
